# Model inventory table

Filter a model-independent element table and export the current result to CSV.

This sample uses only standard SysML v2 concepts and automatically discovers the project's `model/` or `src/` directory.

In [1]:
from pathlib import Path
from collections import Counter, defaultdict
import syside

def find_sysml_root(start=Path.cwd()):
    """Find the nearest model/ or src/ folder containing textual SysML."""
    for directory in (start, *start.parents):
        for folder_name in ('model', 'src'):
            candidate = directory / folder_name
            if candidate.is_dir() and next(candidate.rglob('*.sysml'), None):
                return candidate
    raise FileNotFoundError('No model/ or src/ directory containing .sysml files was found')

SYSML_ROOT = find_sysml_root()
SYSML_FILES = sorted(SYSML_ROOT.rglob('*.sysml'))
model, diagnostics = syside.try_load_model([str(path) for path in SYSML_FILES])
print(f'Loaded {len(SYSML_FILES)} SysML files from {SYSML_ROOT}')

Loaded 42 SysML files from /Users/poojakashyap/sandbox/memo-meta/memo/examples/gpca-pump/model


In [2]:
from html import escape
from IPython.display import HTML, display

KIND_FILTER = ''   # Example: 'PartUsage' or 'Requirement'
NAME_FILTER = ''   # Case-insensitive name fragment
ROW_LIMIT = 250

semantic_elements = (
    list(model.elements(syside.Usage, include_subtypes=True))
    + list(model.elements(syside.Definition, include_subtypes=True))
)

def text(value):
    return str(value) if value else ''

records = []
for item in semantic_elements:
    kind = type(item).__name__
    name = text(item.name or item.declared_name)
    qualified_name = text(item.qualified_name)
    owner = getattr(item, 'owner', None)
    owner_name = text(getattr(owner, 'qualified_name', None) or getattr(owner, 'name', None))
    if KIND_FILTER and KIND_FILTER.casefold() not in kind.casefold():
        continue
    if NAME_FILTER and NAME_FILTER.casefold() not in (name + ' ' + qualified_name).casefold():
        continue
    records.append({'kind': kind, 'name': name, 'qualified_name': qualified_name, 'owner': owner_name})

records.sort(key=lambda row: (row['kind'], row['qualified_name'], row['name']))
visible = records[:ROW_LIMIT]
rows = ''.join(
    '<tr>' + ''.join(f'<td>{escape(row[key])}</td>' for key in ('kind', 'name', 'qualified_name', 'owner')) + '</tr>'
    for row in visible
)
display(HTML(f'''<p><strong>{len(records):,}</strong> matching elements; showing {len(visible):,}.</p>
<div style="max-height:620px;overflow:auto"><table style="font-size:12px"><thead style="position:sticky;top:0;background:white"><tr><th>Kind</th><th>Name</th><th>Qualified name</th><th>Owner</th></tr></thead><tbody>{rows}</tbody></table></div>'''))

Kind,Name,Qualified name,Owner
ActionDefinition,AcquireSensorData,memo_examples_gpca_pump_model_catalog_gpca_behavior_actions::AcquireSensorData,memo_examples_gpca_pump_model_catalog_gpca_behavior_actions
ActionDefinition,ActuatePumpMotor,memo_examples_gpca_pump_model_catalog_gpca_behavior_actions::ActuatePumpMotor,memo_examples_gpca_pump_model_catalog_gpca_behavior_actions
ActionDefinition,ComputeFlowCommand,memo_examples_gpca_pump_model_catalog_gpca_behavior_actions::ComputeFlowCommand,memo_examples_gpca_pump_model_catalog_gpca_behavior_actions
ActionDefinition,EnforceDoseLimits,memo_examples_gpca_pump_model_catalog_gpca_behavior_actions::EnforceDoseLimits,memo_examples_gpca_pump_model_catalog_gpca_behavior_actions
ActionDefinition,EvaluateAlarmConditions,memo_examples_gpca_pump_model_catalog_gpca_behavior_actions::EvaluateAlarmConditions,memo_examples_gpca_pump_model_catalog_gpca_behavior_actions
ActionDefinition,LogTherapyEvent,memo_examples_gpca_pump_model_catalog_gpca_behavior_actions::LogTherapyEvent,memo_examples_gpca_pump_model_catalog_gpca_behavior_actions
ActionUsage,infusionDeliveryFlow,memo_examples_gpca_pump_model_catalog_gpca_behavior_actions::infusionDeliveryFlow,memo_examples_gpca_pump_model_catalog_gpca_behavior_actions
ActionUsage,acquireSensors,memo_examples_gpca_pump_model_catalog_gpca_behavior_actions::infusionDeliveryFlow::acquireSensors,memo_examples_gpca_pump_model_catalog_gpca_behavior_actions::infusionDeliveryFlow
ActionUsage,actuateMotor,memo_examples_gpca_pump_model_catalog_gpca_behavior_actions::infusionDeliveryFlow::actuateMotor,memo_examples_gpca_pump_model_catalog_gpca_behavior_actions::infusionDeliveryFlow
ActionUsage,computeFlow,memo_examples_gpca_pump_model_catalog_gpca_behavior_actions::infusionDeliveryFlow::computeFlow,memo_examples_gpca_pump_model_catalog_gpca_behavior_actions::infusionDeliveryFlow


In [3]:
import csv

OUTPUT = Path('model-inventory.csv')
with OUTPUT.open('w', newline='', encoding='utf-8') as stream:
    writer = csv.DictWriter(stream, fieldnames=['kind', 'name', 'qualified_name', 'owner'])
    writer.writeheader()
    writer.writerows(records)
print(f'Exported {len(records):,} rows to {OUTPUT.resolve()}')

Exported 6,676 rows to /Users/poojakashyap/sandbox/memo-meta/memo/examples/gpca-pump/analysis/Samples/model-inventory.csv
